# Dataset and DataLoader

During batch training, it gets messy when processing data samples and becomes hard to maintain. For this, PyTorch provides two classes Dataset and DataLoader that separates the process of accessing and loading data from training model over loop.

## Dataset Class

The dataset class acts as an interface that knows where data is stored and how it can be retrieved. To implement this class, three essential methods are mentioned:
* `__init__()`
* `__len__()`
* `__getitem__()`

**(i) `__init__()`**

The constructor method is where we define how data should be accessed, how it should be stored (as in features and labels), and how it should be returned (as in defining any transformations).

In other words, it initializes the dataset variables such as file paths, features, labels, transformation for preprocessing (if any), etc.

**(ii) `__len__()`**

This method returns the number of samples in the dataset.

**(iii) `__getitem__()`**

This method fetches and returns a sample from the dataset in the way we defined in the constructor method.

```
from torch.utils.data import Dataset

class CustomDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, index):
    return self.X[index], self.y[index]
```

## DataLoader Class

Now that we have a Dataset which provides a structured way to access individual samples when requested, we need a mechanism to efficiently retrieve and organize those samples for training. In mini-batch training, we require groups of samples (batches) rather than individual samples. This is where the DataLoader class comes in where you tell which dataset to access, what should be the batch size, how samples are retrieved, etc.

**Syntax:**
```
DataLoader(dataset, batch_size=1, shuffle=False, sampler=None,
           batch_sampler=None, num_workers=0, collate_fn=None,
           pin_memory=False, drop_last=False, timeout=0,
           worker_init_fn=None, *, prefetch_factor=2,
           persistent_workers=False)

```

Some important parameters include:
* **dataset** - the dataset to load data from (usually the instance of Dataset)
* **batch_size** - the number of samples in every batch
* **shuffle** - if `True`, samples are randomly ordered after every epoch
* **sampler** - define how samples are drawn; if specified, shuffle parameter cannot be used
* **num_workers** - number of subprocesses required to load data into batches; `0` means data loads in the main process
* **collate_fn** - specifies how samples are combined into a batch
* **pin_memory** - if `True`, DataLoader will copy tensors into pinned memory, which can improve GPU transfer speed
* **drop_last** - if `True`, the last batch containing samples less than the defined batch size is dropped


```
from torch.utils.data import DataLoader

dataset = CustomDataset(X, y)
dataloader = DataLoader(dataset, batch_size = 3, shuffle=True, drop_last = False)
```


# Complete Implementation Example

## Import essential libraries

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from torch.utils.data import Dataset, DataLoader

## Load dataset

In [2]:
import kagglehub
import os

path = kagglehub.dataset_download("srisyra02/online-food-ordering-dataset")
files = os.listdir(path)

print("Path to dataset files:", path)
print("Files:\n", files)

Using Colab cache for faster access to the 'online-food-ordering-dataset' dataset.
Path to dataset files: /kaggle/input/online-food-ordering-dataset
Files:
 ['online food delivery dataset.csv']


In [3]:
pd.set_option('display.max_columns', None)
data = pd.read_csv(os.path.join(path, files[0]),)
data.head(10)

,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,Customer Type,latitude,longitude,Pin code,Output,Feedback,Unnamed: 13
0,20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,560001,Yes,Positive,Yes
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.9770,77.5773,560009,Yes,Positive,Yes
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,560017,Yes,Negative,Yes
3,22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,560019,Yes,Positive,Yes
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.9850,77.5533,560010,Yes,Positive,Yes
5,27,Female,Married,Employee,More than 50000,Post Graduate,2,Regular,12.9299,77.6848,560103,Yes,Positive,Yes
6,22,Male,Single,Student,No Income,Graduate,3,Regular,12.9770,77.5773,560009,Yes,Positive,Yes
7,24,Female,Single,Student,No Income,Post Graduate,3,Regular,12.9828,77.6131,560042,Yes,Positive,Yes
8,23,Female,Single,Student,No Income,Post Graduate,2,Regular,12.9766,77.5993,560001,Yes,Positive,Yes
9,23,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9854,77.7081,560048,Yes,Positive,Yes


## Gather basic information

In [4]:
data.columns

Index(['Age', 'Gender', 'Marital Status', 'Occupation', 'Monthly Income',
       'Educational Qualifications', 'Family size', 'Customer Type',
       'latitude', 'longitude', 'Pin code', 'Output', 'Feedback',
       'Unnamed: 13'],
      dtype='object')

In [5]:
data.drop(columns=['Pin code', 'Unnamed: 13'], inplace=True) # for simplicity
data.head(5)

,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,Customer Type,latitude,longitude,Output,Feedback
0,20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,Yes,Positive
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.9770,77.5773,Yes,Positive
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,Yes,Negative
3,22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,Yes,Positive
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.9850,77.5533,Yes,Positive


In [6]:
df = data.copy()
df.head()

,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,Customer Type,latitude,longitude,Output,Feedback
0,20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,Yes,Positive
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.9770,77.5773,Yes,Positive
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,Yes,Negative
3,22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,Yes,Positive
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.9850,77.5533,Yes,Positive


In [7]:
df.columns = df.columns.str.replace(r'(?<=\S)\s+(?=\S)', '_', regex=True).str.lower()
df.head()

,age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size,customer_type,latitude,longitude,output,feedback
0,20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,Yes,Positive
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.9770,77.5773,Yes,Positive
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,Yes,Negative
3,22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,Yes,Positive
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.9850,77.5533,Yes,Positive


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 388 entries, 0 to 387
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   age                         388 non-null    int64  
 1   gender                      388 non-null    object 
 2   marital_status              388 non-null    object 
 3   occupation                  388 non-null    object 
 4   monthly_income              388 non-null    object 
 5   educational_qualifications  388 non-null    object 
 6   family_size                 388 non-null    int64  
 7   customer_type               388 non-null    object 
 8   latitude                    388 non-null    float64
 9   longitude                   388 non-null    float64
 10  output                      388 non-null    object 
 11  feedback                    388 non-null    object 
dtypes: float64(2), int64(2), object(8)
memory usage: 36.5+ KB


In [9]:
print('Shape before removing duplicates: ',df.shape)
if df.duplicated().sum() != np.int64(0) :
  df.drop_duplicates(inplace=True)
print('Shape after removing duplicates: ',df.shape)

Shape before removing duplicates:  (388, 12)
Shape after removing duplicates:  (285, 12)


In [10]:
df.head()

,age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size,customer_type,latitude,longitude,output,feedback
0,20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,Yes,Positive
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.9770,77.5773,Yes,Positive
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,Yes,Negative
3,22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,Yes,Positive
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.9850,77.5533,Yes,Positive


## Split data into train and test

In [11]:
X = df.drop(columns=['output'])
y = df['output']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Basic preprocessing

In [12]:
for col in df.columns:
  if df[col].dtype == 'O':
    print(col+': ',df[col].unique())

gender:  ['Female' 'Male']
marital_status:  ['Single' 'Married' 'Prefer not to say']
occupation:  ['Student' 'Employee' 'Self Employeed' 'House wife']
monthly_income:  ['No Income' 'Below Rs.10000' 'More than 50000' '10001 to 25000'
 '25001 to 50000']
educational_qualifications:  ['Post Graduate' 'Graduate' 'Ph.D' 'Uneducated' 'School']
customer_type:  ['Frequent' 'Regular' 'New']
output:  ['Yes' 'No']
feedback:  ['Positive' 'Negative ']


In [13]:
ohe_cols = ['gender', 'marital_status', 'occupation', 'educational_qualifications', 'customer_type', 'feedback']
oe_cols = ['monthly_income']
oe_cat = ['No Income', 'Below Rs.10000', '10001 to 25000', '25001 to 50000', 'More than 50000']
num_cols = ['age', 'family_size', 'latitude', 'longitude']

ct = ColumnTransformer(transformers=[('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), ohe_cols),
                                     ('oe', OrdinalEncoder(categories=[oe_cat], handle_unknown='use_encoded_value', unknown_value=-1), oe_cols),
                                     ('scale', StandardScaler(), num_cols)],
                       remainder='passthrough',
                       verbose_feature_names_out=False)

le_col = ['output']

le = LabelEncoder()

In [14]:
X_train_proc = ct.fit_transform(X_train)
X_test_proc = ct.transform(X_test)

y_train_proc = le.fit_transform(y_train)
y_test_proc = le.transform(y_test)

In [15]:
X_train_tensor = torch.from_numpy(X_train_proc).float()
X_test_tensor = torch.from_numpy(X_test_proc).float()

y_train_tensor = torch.from_numpy(y_train_proc).view(-1, 1).float()
y_test_tensor = torch.from_numpy(y_test_proc).view(-1, 1).float()

print(X_train_tensor.shape, X_train_tensor.dtype)
print(X_test_tensor.shape, X_test_tensor.dtype)
print(y_train_tensor.shape, y_train_tensor.dtype)
print(y_test_tensor.shape, y_test_tensor.dtype)

torch.Size([228, 18]) torch.float32
torch.Size([57, 18]) torch.float32
torch.Size([228, 1]) torch.float32
torch.Size([57, 1]) torch.float32


## Create CustomDataset object for train and test data

In [16]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
  def __init__(self, features, target):
    self.features = features
    self.target = target

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.target[index]


train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)


**Note:** Instead of having to write this CustomDataset class manually, PyTorch provides a built-in class called 'TensorDataset' which implements the same logic as Dataset. So an alternative way of doing this is:

```
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
```

## Create DataLoader object for train and test data

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, drop_last=False)
# shuffle is set to False while creating test_dataloader because the order doesn't matter during evaluation

## Construct a neural network model

In [18]:
import torch.nn as nn

class SimpleNNModel(nn.Module):
  def __init__(self, num_features):

    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(num_features, 5),
        nn.ReLU(),
        nn.Linear(5, 1),
        nn.Sigmoid()
    )

  def forward(self, features):
    out = self.network(features)
    return out

## Train model in batches

In [19]:
my_model = SimpleNNModel(X_train_tensor.shape[1])

loss_func = nn.BCELoss()

optimizer = torch.optim.Adam(my_model.parameters(), lr=0.01)

epochs = 50

for epoch in range(epochs):
  loss_ls = []
  for batch_features, batch_target in train_dataloader:
    y_pred = my_model(batch_features)
    loss = loss_func(y_pred, batch_target)
    loss_ls.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  epoch_loss = sum(loss_ls)/len(loss_ls)
  if epoch % 5 == 0:
    print(f'Epoch: {epoch}, Loss: {epoch_loss:.4f}')

Epoch: 0, Loss: 0.5664
Epoch: 5, Loss: 0.4563
Epoch: 10, Loss: 0.3929
Epoch: 15, Loss: 0.3691
Epoch: 20, Loss: 0.3397
Epoch: 25, Loss: 0.3492
Epoch: 30, Loss: 0.3265
Epoch: 35, Loss: 0.3269
Epoch: 40, Loss: 0.3164
Epoch: 45, Loss: 0.2959


## Make predictions and evaluate model

During evaluation, we explicitly set the model to evaluation mode using `model.eval()`. This is necessary because certain layers behave differently during training and evaluation.

For example, during training, dropout randomly deactivates some neurons to prevent overfitting. However, during evaluation or prediction, dropout is disabled, and all neurons are used. Similarly in batch Normalization training, it computes the mean and variance from the current mini-batch. It also maintains running estimates of these statistics. During evaluation, instead of computing statistics from the current batch, it uses the running mean and standard deviation learned during training.

Therefore, setting the model to evaluation mode ensures that layers such as dropout and batch normalization behave appropriately when making predictions.


In [20]:
# set model in evaluation mode
my_model.eval()

with torch.no_grad():
  accuracy_ls = []
  for batch_features, batch_target in test_dataloader:
    y_pred_t = my_model(batch_features)
    y_pred_t = (y_pred_t > 0.5).float()
    accuracy = (y_pred_t == batch_target).float().mean()
    accuracy_ls.append(accuracy.item())
  accuracy = sum(accuracy_ls)/len(accuracy_ls)
  print(f'Accuracy: {accuracy}')

Accuracy: 0.8975000083446503
